# 05 — Análise Exploratória de Dados (EDA)

**Objetivo:** gerar as visualizações e estatísticas exploratórias que sustentam as hipóteses H1, H2 e H3 do artigo antes da modelagem preditiva.

## Hipóteses verificadas

| Hipótese | Análise | Figura |
|----------|---------|--------|
| **H1** — Desigualdade regional sistemática | Boxplot notas por macrorregião | `fig_boxplot_regioes.png` |
| **H2** — Renda familiar correlaciona com desempenho (varia por região) | Heatmap Spearman renda × nota | `fig_heatmap_spearman.png` |
| **H3** — IDHM estadual explica variação além da renda individual | Scatter IDHM × nota municipal | `fig_scatter_idhm_nota.png` |

## Estratégia de acesso aos dados

Este notebook usa **DuckDB in-memory** lendo diretamente o `dataset_analitico.parquet` — sem abrir o arquivo `.duckdb`. Isso evita conflito de lock com o notebook 04 e permite rodar os dois simultaneamente.

> ⚠️ Todas as colunas de nota e indicadores são `TEXT` no Parquet — usar `CAST(col AS DOUBLE)` em todas as queries numéricas.

**Figuras geradas em:** `data/processed/`

## 1. Imports e configuração

In [1]:
import duckdb
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
from scipy import stats
from pathlib import Path

Path('../data/processed').mkdir(parents=True, exist_ok=True)

# Lê direto do parquet — sem travar o arquivo .duckdb
PARQUET = '../data/processed/dataset_analitico.parquet'
con = duckdb.connect()  # in-memory
con.execute(f"CREATE VIEW dataset_analitico AS SELECT * FROM read_parquet('{PARQUET}')")

plt.rcParams.update({'figure.dpi': 120, 'font.size': 10})
REGIOES = ['Norte','Nordeste','Centro-Oeste','Sudeste','Sul']
CORES   = {'Norte':'#e07b39','Nordeste':'#e0c239','Centro-Oeste':'#39a85e',
           'Sudeste':'#3980e0','Sul':'#8e39e0'}


## 5.1 — Boxplot: notas por macrorregião (evidência H1)

Compara a distribuição de notas de Matemática e Redação entre as cinco macrorregiões brasileiras.

**Amostra:** 200k candidatos aleatórios via `USING SAMPLE` do DuckDB — suficiente para visualização sem sobrecarregar memória.

**Interpretação esperada:** Sudeste e Sul com medianas e quartis superiores sistematicamente acima de Norte e Nordeste — confirmando H1. Notas com notch (`notch=True`) permitem comparar intervalos de confiança das medianas visualmente.

In [ ]:
df_box = con.execute("""
    SELECT REGIAO,
           CAST(NU_NOTA_MT AS DOUBLE)      AS NU_NOTA_MT,
           CAST(NU_NOTA_REDACAO AS DOUBLE) AS NU_NOTA_REDACAO
    FROM dataset_analitico
    WHERE REGIAO IS NOT NULL AND NU_NOTA_MT IS NOT NULL
    USING SAMPLE 200000
""").df()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, titulo, col in zip(axes, ['Matemática','Redação'],['NU_NOTA_MT','NU_NOTA_REDACAO']):
    data = [df_box[df_box['REGIAO']==r][col].dropna() for r in REGIOES]
    bp = ax.boxplot(data, labels=REGIOES, patch_artist=True, notch=True,
                    medianprops={'color':'white','linewidth':2})
    for patch, reg in zip(bp['boxes'], REGIOES): patch.set_facecolor(CORES[reg])
    ax.set_title(f'Nota {titulo} por Macrorregião'); ax.set_ylabel('Nota'); ax.grid(axis='y',alpha=0.3)
plt.tight_layout()
plt.savefig('../data/processed/fig_boxplot_regioes.png', bbox_inches='tight')
plt.show()


## 5.2 — Heatmap Spearman: renda × nota por região (evidência H2)

Calcula o coeficiente de correlação de Spearman entre a **renda familiar harmonizada** (`Q006_HARM`, codificada como ordinal 0–4) e as notas em cada área, para cada macrorregião separadamente.

**Por que Spearman?** `Q006_HARM` é uma escala ordinal (A < B < C < D < E), não intervalar. Spearman é mais adequado que Pearson para este tipo de variável.

**Amostra:** 300k candidatos para garantir estabilidade dos coeficientes.

**Interpretação esperada:**
- Correlações positivas em todas as regiões (H2 confirmada)
- Magnitude variando por região (H2 — o efeito não é uniforme)
- Correlações geralmente fracas (ρ < 0.15) — justifica incluir IDHM como variável contextual adicional

> **Limitação:** `Q006_HARM` captura a renda declarada pelo candidato, não a renda do município. Isso subestima o papel do contexto socioeconômico — motivação para incluir o IDHM nos modelos M2/M3.

In [ ]:
df_corr = con.execute("""
    SELECT REGIAO, Q006_HARM,
           CAST(NU_NOTA_MT AS DOUBLE) AS NU_NOTA_MT,
           CAST(NU_NOTA_CN AS DOUBLE) AS NU_NOTA_CN,
           CAST(NU_NOTA_CH AS DOUBLE) AS NU_NOTA_CH,
           CAST(NU_NOTA_LC AS DOUBLE) AS NU_NOTA_LC,
           CAST(NU_NOTA_REDACAO AS DOUBLE) AS NU_NOTA_REDACAO
    FROM dataset_analitico
    WHERE REGIAO IS NOT NULL AND Q006_HARM IS NOT NULL
    USING SAMPLE 300000
""").df()

q006_ord = {'A':0,'B':1,'C':2,'D':3,'E':4}
df_corr['Q006_NUM'] = df_corr['Q006_HARM'].map(q006_ord)
AREAS = ['MT','CN','CH','LC','REDACAO']

matriz = {}
for reg in REGIOES:
    sub = df_corr[df_corr['REGIAO']==reg]
    if len(sub) < 100: continue
    matriz[reg] = {a: stats.spearmanr(sub['Q006_NUM'], sub[f'NU_NOTA_{a}'], nan_policy='omit').statistic for a in AREAS}

df_hm = pd.DataFrame(matriz).T
df_hm.columns = ['Mat','CN','CH','LC','Redação']

fig, ax = plt.subplots(figsize=(8, 4))
im = ax.imshow(df_hm.values, cmap='RdYlGn', vmin=-0.1, vmax=0.6, aspect='auto')
ax.set_xticks(range(len(df_hm.columns))); ax.set_xticklabels(df_hm.columns)
ax.set_yticks(range(len(df_hm.index)));   ax.set_yticklabels(df_hm.index)
for i in range(len(df_hm.index)):
    for j in range(len(df_hm.columns)):
        ax.text(j, i, f'{df_hm.values[i,j]:.2f}', ha='center', va='center', fontsize=10)
plt.colorbar(im, ax=ax, label='Spearman ρ')
ax.set_title('Correlação Spearman: renda familiar × nota')
plt.tight_layout()
plt.savefig('../data/processed/fig_heatmap_spearman.png', bbox_inches='tight')
plt.show()


## 5.3 — Distribuição das faixas de nota (verifica desbalanceamento)

Mostra o percentual de candidatos em cada faixa (`<400` a `>800`) para três métricas: Matemática, Redação e Média Geral (CN+CH+LC+MT+Redação ÷ 5).

**Por que isso importa para os modelos?**  
Classes muito desbalanceadas (ex: `>800` com <2%) exigem estratégia específica no Random Forest. Se alguma faixa tiver menos de 5%, usar `class_weight='balanced'` e documentar no artigo.

A Média Geral é calculada diretamente na query SQL para evitar carregar todas as notas em memória.

In [ ]:
# Faixas para MT, Redação e média geral (CN+CH+LC+MT+REDACAO)
df_notas = con.execute("""
    SELECT
        FAIXA_MT,
        FAIXA_REDACAO,
        CASE
            WHEN CAST(NU_NOTA_CN AS DOUBLE) IS NULL THEN NULL
            ELSE CASE
                WHEN (CAST(NU_NOTA_CN AS DOUBLE) + CAST(NU_NOTA_CH AS DOUBLE) +
                      CAST(NU_NOTA_LC AS DOUBLE) + CAST(NU_NOTA_MT AS DOUBLE) +
                      CAST(NU_NOTA_REDACAO AS DOUBLE)) / 5.0 < 400  THEN '<400'
                WHEN (CAST(NU_NOTA_CN AS DOUBLE) + CAST(NU_NOTA_CH AS DOUBLE) +
                      CAST(NU_NOTA_LC AS DOUBLE) + CAST(NU_NOTA_MT AS DOUBLE) +
                      CAST(NU_NOTA_REDACAO AS DOUBLE)) / 5.0 < 500  THEN '400-500'
                WHEN (CAST(NU_NOTA_CN AS DOUBLE) + CAST(NU_NOTA_CH AS DOUBLE) +
                      CAST(NU_NOTA_LC AS DOUBLE) + CAST(NU_NOTA_MT AS DOUBLE) +
                      CAST(NU_NOTA_REDACAO AS DOUBLE)) / 5.0 < 600  THEN '500-600'
                WHEN (CAST(NU_NOTA_CN AS DOUBLE) + CAST(NU_NOTA_CH AS DOUBLE) +
                      CAST(NU_NOTA_LC AS DOUBLE) + CAST(NU_NOTA_MT AS DOUBLE) +
                      CAST(NU_NOTA_REDACAO AS DOUBLE)) / 5.0 < 700  THEN '600-700'
                WHEN (CAST(NU_NOTA_CN AS DOUBLE) + CAST(NU_NOTA_CH AS DOUBLE) +
                      CAST(NU_NOTA_LC AS DOUBLE) + CAST(NU_NOTA_MT AS DOUBLE) +
                      CAST(NU_NOTA_REDACAO AS DOUBLE)) / 5.0 < 800  THEN '700-800'
                ELSE '>800'
            END
        END AS FAIXA_GERAL
    FROM dataset_analitico
    WHERE FAIXA_MT IS NOT NULL
""").df()

FAIXAS_ORD = ['<400','400-500','500-600','600-700','700-800','>800']
SERIES = [
    ('FAIXA_MT',      'Matemática',    '#3980e0'),
    ('FAIXA_REDACAO', 'Redação',        '#e07b39'),
    ('FAIXA_GERAL',   'Média Geral',    '#39a85e'),
]

fig, axes = plt.subplots(1, 3, figsize=(16, 5), sharey=False)
for ax, (col, titulo, cor) in zip(axes, SERIES):
    counts = df_notas[col].value_counts().reindex(FAIXAS_ORD, fill_value=0)
    pcts   = 100 * counts / counts.sum()
    ax.bar(FAIXAS_ORD, pcts, color=cor, edgecolor='white')
    ax.yaxis.set_major_formatter(mtick.PercentFormatter())
    ax.set_title(f'Faixas — {titulo}')
    ax.set_xlabel('Faixa de nota')
    ax.set_ylabel('%')
    ax.tick_params(axis='x', rotation=30)
    for i, (faixa, pct) in enumerate(zip(FAIXAS_ORD, pcts)):
        ax.text(i, pct + 0.3, f'{pct:.1f}%', ha='center', fontsize=8)

plt.suptitle('Distribuição de faixas de nota (2012–2024)', y=1.02)
plt.tight_layout()
plt.savefig('../data/processed/fig_faixas_notas.png', bbox_inches='tight')
plt.show()

# Tabela resumo
resumo = pd.DataFrame({
    t: (100 * df_notas[c].value_counts() / len(df_notas)).reindex(FAIXAS_ORD).round(2)
    for c, t, _ in SERIES
})
print(resumo.to_string())


## 5.4 — Scatter: IDHM × nota média municipal (evidência H3)

Agrega os candidatos por município (mínimo 30 candidatos para estabilidade estatística) e plota o IDHM médio do estado contra a nota média do município — para Média Geral, Matemática e Redação simultaneamente.

**Por que três subplots?** Cada área de conhecimento pode ter relação diferente com o IDHM. Redação, por exemplo, pode ser mais influenciada por fatores culturais (acesso a leitura, qualidade do ensino de língua portuguesa) do que Matemática.

O tamanho de cada ponto é proporcional ao número de candidatos do município (até 5.000 limitado) — municípios maiores têm estimativas mais confiáveis.

**Interpretação esperada:** correlação positiva moderada a alta (ρ > 0.45) em todas as notas — confirmando H3. Municípios do Norte/Nordeste (laranja/amarelo) agrupados no canto inferior esquerdo; Sul/Sudeste (roxo/azul) no superior direito.

> **Limitação importante:** o IDHM aqui é estadual (mesmo valor para todos os municípios de um estado). A correlação observada reflete variação **entre** estados, não dentro deles. Com dados municipais reais, espera-se correlação ainda mais forte.

In [ ]:
df_scatter = con.execute("""
    SELECT
        CO_MUNICIPIO_ESC,
        REGIAO,
        AVG(CAST(idhm AS DOUBLE)) AS idhm_medio,
        AVG(CAST(NU_NOTA_MT AS DOUBLE)) AS media_mt,
        AVG(CAST(NU_NOTA_REDACAO AS DOUBLE)) AS media_redacao,
        AVG((CAST(NU_NOTA_CN AS DOUBLE) + CAST(NU_NOTA_CH AS DOUBLE) +
             CAST(NU_NOTA_LC AS DOUBLE) + CAST(NU_NOTA_MT AS DOUBLE) +
             CAST(NU_NOTA_REDACAO AS DOUBLE)) / 5.0) AS media_geral,
        COUNT(*) AS n_candidatos
    FROM dataset_analitico
    WHERE idhm IS NOT NULL AND NU_NOTA_MT IS NOT NULL AND REGIAO IS NOT NULL
    GROUP BY CO_MUNICIPIO_ESC, REGIAO
    HAVING COUNT(*) >= 30
""").df()

SERIES = [
    ('media_geral',   'Média Geral',  '#39a85e'),
    ('media_mt',      'Matemática',   '#3980e0'),
    ('media_redacao', 'Redação',      '#e07b39'),
]

fig, axes = plt.subplots(1, 3, figsize=(18, 6), sharey=False)

for ax, (col, titulo, _) in zip(axes, SERIES):
    for reg in REGIOES:
        sub = df_scatter[df_scatter['REGIAO'] == reg]
        ax.scatter(sub['idhm_medio'], sub[col],
                   s=sub['n_candidatos'].clip(upper=5000) / 50,
                   alpha=0.5, color=CORES[reg], label=reg, edgecolors='none')
    # Linha de tendência
    valid = df_scatter[['idhm_medio', col]].dropna()
    m, b  = np.polyfit(valid['idhm_medio'], valid[col], 1)
    x     = np.linspace(valid['idhm_medio'].min(), valid['idhm_medio'].max(), 100)
    ax.plot(x, m*x + b, 'k--', linewidth=1.5)
    rho, pval = stats.spearmanr(valid['idhm_medio'], valid[col])
    ax.set_xlabel('IDHM médio do município')
    ax.set_ylabel('Nota média')
    ax.set_title(f'IDHM × {titulo}\nρ={rho:.3f}  p={pval:.1e}')
    ax.grid(alpha=0.2)

# Legenda única
handles = [plt.scatter([], [], color=CORES[r], label=r, alpha=0.7) for r in REGIOES]
fig.legend(handles=handles, title='Macrorregião',
           loc='lower center', ncol=5, bbox_to_anchor=(0.5, -0.08))

plt.suptitle('IDHM municipal × Nota média por município', y=1.02)
plt.tight_layout()
plt.savefig('../data/processed/fig_scatter_idhm_nota.png', bbox_inches='tight')
plt.show()
